In [ ]:

import subprocess, sys

packages = [
    'python-dotenv',   # Safe .env loading
    'requests',        # HTTP calls to OpenRouter
    'pandas',          # Dataframe manipulation
    'numpy',           # Numeric ops
    'matplotlib',      # Plotting
    'seaborn',         # Statistical plots
    'datasets',        # HuggingFace datasets
    'transformers',    # Tokenizer (flan-t5-base)
    'rouge-score',     # ROUGE metrics
    'sentence-transformers',  # Semantic similarity
    'ipywidgets',      # Interactive demo widget
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

import os, re, time, json, textwrap
import requests as http_requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoTokenizer
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util as st_util
from sklearn.model_selection import train_test_split

load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')
assert API_KEY, '❌ OPENROUTER_API_KEY not found — create a .env file with your key'

sns.set_palette('viridis')



In [ ]:


def degrade_prompt(good_prompt):
    """Strip structure from a well-crafted prompt to simulate a weak one."""
    text = good_prompt.lower().strip()
    text = re.sub(r'(act as|you are|as a|as an|i want you to)\s+\w+', '', text)
    text = re.sub(r'[\-\*•]\s+', '', text)
    text = re.sub(r'\d+\.\s+', '', text)
    text = re.sub(r'(in bullet points|step by step|in detail|please|provide|explain)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    return ' '.join(words[:min(8, max(3, len(words) // 4))])

hf_data = load_dataset('fka/awesome-chatgpt-prompts', split='train')
source_a = pd.DataFrame({
    'weak_prompt': [degrade_prompt(row['prompt']) for row in hf_data],
    'improved_prompt': [row['prompt'] for row in hf_data],
    'source': 'huggingface'
})


SEED_TOPICS = [
    'explain ai', 'python basics', 'climate change', 'healthy eating',
    'machine learning', 'web development', 'photography tips', 'time management',
    'space exploration', 'financial planning', 'creative writing', 'data science',
    'public speaking', 'mental health', 'cybersecurity', 'blockchain',
    'remote work', 'renewable energy', 'artificial intelligence ethics', 'fitness',
    'resume writing', 'social media marketing', 'cloud computing', 'game design',
    'travel planning', 'cooking', 'mindfulness', 'startup ideas',
    'language learning', 'UX design', 'content strategy', 'email writing',
    'project management', 'negotiation skills', 'database design', 'networking',
    'mobile app development', 'video editing', 'SEO optimization', 'podcasting',
]

SYSTEM_PROMPT = """You are an expert AI prompt engineer. Transform the following weak prompt into a 
highly effective, structured prompt. Include: a clear ROLE, specific ACTION, relevant CONTEXT, 
desired OUTPUT FORMAT, and SPECIFICITY. Return ONLY the improved prompt text."""

OPENROUTER_MODEL = 'mistralai/mistral-small-creative'

def call_openrouter(prompt, system=SYSTEM_PROMPT, max_tokens=300):
    """Call OpenRouter API and return the generated text."""
    try:
        resp = http_requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {API_KEY}',
                'Content-Type': 'application/json',
                'HTTP-Referer': 'https://prompt-optimizer.vercel.app',
                'X-Title': 'Prompt Optimizer Notebook',
            },
            json={
                'model': OPENROUTER_MODEL,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                'max_tokens': max_tokens,
                'temperature': 0.7,
            },
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content'].strip()
    except Exception as e:
        return None  # skip failed calls

source_b_rows = []
    improved = call_openrouter(topic)
    if improved and len(improved.split()) >= 15:
        source_b_rows.append({
            'weak_prompt': topic,
            'improved_prompt': improved,
            'source': 'openrouter'
        })
    time.sleep(1.0)  # rate-limit courtesy (free tier)

source_b = pd.DataFrame(source_b_rows)

df = pd.concat([source_a, source_b], ignore_index=True)
df.head()

In [ ]:


df.dropna(subset=['weak_prompt', 'improved_prompt'], inplace=True)
df.drop_duplicates(subset=['weak_prompt'], inplace=True)

def clean_text(text):
    text = re.sub(r'<[^>]+>', '', str(text))        # strip HTML tags
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)     # remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()        # collapse whitespace
    return text

df['weak_prompt']     = df['weak_prompt'].apply(lambda x: clean_text(x).lower())
df['improved_prompt'] = df['improved_prompt'].apply(clean_text)

df['weak_len']     = df['weak_prompt'].apply(lambda x: len(x.split()))
df['improved_len'] = df['improved_prompt'].apply(lambda x: len(x.split()))

df = df[(df['weak_len'] >= 2) & (df['weak_len'] <= 10) &
        (df['improved_len'] >= 15) & (df['improved_len'] <= 300)].copy()

df['input_text']  = 'improve prompt: ' + df['weak_prompt']
df['target_text'] = df['improved_prompt']

df.reset_index(drop=True, inplace=True)

df[['weak_prompt', 'improved_prompt', 'source']].sample(min(5, len(df)), random_state=42)

In [ ]:

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

input_encodings  = tokenizer(df['input_text'].tolist(),  padding=True, truncation=True, max_length=128, return_tensors='np')
target_encodings = tokenizer(df['target_text'].tolist(), padding=True, truncation=True, max_length=256, return_tensors='np')

input_token_lens  = (input_encodings['attention_mask'].sum(axis=1)).tolist()
target_token_lens = (target_encodings['attention_mask'].sum(axis=1)).tolist()


axes[0].hist(input_token_lens, bins=30, color='#818cf8', edgecolor='#1e1b4b', alpha=0.85)
axes[0].set_title('Input Token Lengths', fontweight='bold')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Count')

axes[1].hist(target_token_lens, bins=30, color='#4ade80', edgecolor='#052e16', alpha=0.85)
axes[1].set_title('Target Token Lengths', fontweight='bold')
axes[1].set_xlabel('Tokens'); axes[1].set_ylabel('Count')


In [ ]:

df['length_bucket'] = pd.cut(
    df['weak_len'],
    bins=[0, 3, 6, 100],
    labels=['short', 'medium', 'long']
)

bucket_counts = df['length_bucket'].value_counts()
can_stratify = all(c >= 2 for c in bucket_counts.values)
strat_col = df['length_bucket'] if can_stratify else None

train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=strat_col
)

strat_temp = temp_df['length_bucket'] if can_stratify else None
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=strat_temp
)


split_counts = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Count': [len(train_df), len(val_df), len(test_df)]
})
              color=['#6366f1', '#a78bfa', '#c4b5fd'], edgecolor='#1e1b4b')
for bar, count in zip(bars, split_counts['Count']):
            str(count), ha='center', fontweight='bold', color='white')


In [ ]:

def score_prompt(prompt):
    """Rate a prompt 0–10 on length, role, clarity, format, specificity."""
    text = str(prompt).strip()
    if not text:
        return 0
    words = text.split()
    score = 0

    if len(words) >= 20: score += 2
    elif len(words) >= 10: score += 1

    role_kw = ['act as', 'you are', 'as a', 'as an', 'role:', 'persona:']
    if any(k in text.lower() for k in role_kw): score += 2

    actions = ['explain', 'describe', 'list', 'compare', 'write', 'create',
               'summarize', 'analyze', 'generate', 'give me', 'provide', 'show']
    if '?' in text or any(v in text.lower() for v in actions): score += 2
    else: score += 1

    fmt = ['bullet', 'list', 'step', 'table', 'json', 'markdown',
           'numbered', 'format', 'structure', 'outline', 'paragraph']
    if any(f in text.lower() for f in fmt): score += 2

    has_nums = bool(re.search(r'\d', text))
    has_proper = bool(re.search(r'[A-Z]', text[1:])) if len(text) > 1 else False
    if has_nums or (has_proper and len(words) >= 15): score += 2
    elif len(words) >= 15: score += 1

    return score


api_outputs = []
    result = call_openrouter(row['weak_prompt'])
    api_outputs.append(result if result else '')  # empty string on failure
    time.sleep(1.0)  # rate-limit courtesy for free tier

train_df = train_df.copy()
train_df['api_output'] = api_outputs

valid_mask = train_df['api_output'].str.len() > 0
train_df = train_df[valid_mask].copy()

if len(train_df) == 0:
    raise RuntimeError('❌ All API calls failed — check your OPENROUTER_API_KEY in .env')

train_df['score_original']     = train_df['weak_prompt'].apply(score_prompt)
train_df['score_ground_truth'] = train_df['improved_prompt'].apply(score_prompt)
train_df['score_api_output']   = train_df['api_output'].apply(score_prompt)

train_df.to_csv('results.csv', index=False)


In [ ]:

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for _, row in train_df.iterrows():
    scores = scorer.score(row['improved_prompt'], row['api_output'])
    for key in rouge_results:
        rouge_results[key].append(scores[key].fmeasure)

rouge_means = {k: np.mean(v) if len(v) > 0 else 0.0 for k, v in rouge_results.items()}

              color=['#6366f1', '#8b5cf6', '#a78bfa'], edgecolor='#1e1b4b')
for bar, val in zip(bars, rouge_means.values()):
            f'{val:.4f}', ha='center', fontweight='bold', color='white', fontsize=11)
max_rouge = max(rouge_means.values()) if rouge_means.values() else 0

sem_model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight, CPU-friendly

gt_embeddings  = sem_model.encode(train_df['improved_prompt'].tolist(), show_progress_bar=True)
api_embeddings = sem_model.encode(train_df['api_output'].tolist(), show_progress_bar=True)

cos_sims = [float(st_util.cos_sim(gt_embeddings[i], api_embeddings[i])) for i in range(len(gt_embeddings))]
train_df['semantic_similarity'] = cos_sims

mean_sim = np.mean(cos_sims) if cos_sims else 0.0

if cos_sims:

score_means = {
    'Original':     train_df['score_original'].mean(),
    'Ground Truth': train_df['score_ground_truth'].mean(),
    'API Output':   train_df['score_api_output'].mean(),
}

colors = ['#ef4444', '#22c55e', '#6366f1']
for bar, val in zip(bars, score_means.values()):
            f'{val:.2f}', ha='center', fontweight='bold', color='white', fontsize=12)

comparison = train_df[['weak_prompt', 'improved_prompt', 'api_output',
                        'score_original', 'score_api_output']].head(10).copy()
comparison.columns = ['Weak Prompt', 'Ground Truth', 'API Output', 'Score Before', 'Score After']
for col in ['Ground Truth', 'API Output']:
    comparison[col] = comparison[col].apply(lambda x: textwrap.shorten(str(x), width=80, placeholder='…'))
comparison

In [ ]:

avg_gain = (train_df['score_api_output'] - train_df['score_original']).mean()
total_samples = len(df)

train_df['score_gain'] = train_df['score_api_output'] - train_df['score_original']
best_topic  = train_df.loc[train_df['score_gain'].idxmax(), 'weak_prompt']
worst_topic = train_df.loc[train_df['score_gain'].idxmin(), 'weak_prompt']

report = f"""
═══════════════════════════════════════
       PROMPT OPTIMIZER — RESULTS
═══════════════════════════════════════
Dataset        : {total_samples} samples
Train/Val/Test : {len(train_df)} / {len(val_df)} / {len(test_df)}

ROUGE-1        : {rouge_means['rouge1']:.4f}
ROUGE-2        : {rouge_means['rouge2']:.4f}
ROUGE-L        : {rouge_means['rougeL']:.4f}
Semantic Sim   : {mean_sim:.4f}
Avg Score Gain : +{avg_gain:.1f} points (out of 10)

Best performing topic  : {best_topic}
Worst performing topic : {worst_topic}
═══════════════════════════════════════
"""

report_data = pd.DataFrame({
    'Metric': ['Dataset Size', 'Train', 'Validation', 'Test',
               'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Semantic Similarity',
               'Avg Score Gain', 'Best Topic', 'Worst Topic'],
    'Value': [total_samples, len(train_df), len(val_df), len(test_df),
              f"{rouge_means['rouge1']:.4f}", f"{rouge_means['rouge2']:.4f}",
              f"{rouge_means['rougeL']:.4f}", f'{mean_sim:.4f}',
              f'+{avg_gain:.1f}', best_topic, worst_topic]
})
report_data.to_csv('report.csv', index=False)


In [ ]:

import ipywidgets as widgets
from IPython.display import display, HTML

prompt_input = widgets.Textarea(
    value='',
    placeholder='Type a weak prompt, e.g. "explain ai"',
    description='Prompt:',
    layout=widgets.Layout(width='90%', height='80px')
)

output_area = widgets.Output()

def on_optimize(btn):
    output_area.clear_output()
    weak = prompt_input.value.strip()
    if not weak:
        with output_area:
        return

    with output_area:
        improved = call_openrouter(weak)
        if not improved:
            return

        before = score_prompt(weak)
        after  = score_prompt(improved)

        display(HTML(f"""
        <div style="background:#1e1b4b; padding:16px; border-radius:12px; margin-top:8px; font-family:monospace;">
            <div style="color:#a5b4fc; font-weight:bold; margin-bottom:8px;">🔹 Original (Score: {before}/10)</div>
            <div style="color:#9ca3af; margin-bottom:16px;">{weak}</div>
            <div style="color:#4ade80; font-weight:bold; margin-bottom:8px;">🚀 Optimized (Score: {after}/10)</div>
            <div style="color:#e5e7eb;">{improved}</div>
            <div style="margin-top:12px; color:#f59e0b; font-weight:bold;">Score gain: +{after - before} points</div>
        </div>
        """))

optimize_btn = widgets.Button(
    description='🚀 Optimize',
    button_style='info',
    layout=widgets.Layout(width='200px', height='36px')
)
optimize_btn.on_click(on_optimize)

display(HTML('<h3 style="color:#818cf8;">🎯 Live Demo — Try it yourself</h3>'))
display(prompt_input)
display(optimize_btn)
display(output_area)